# VayuSwarm — Behavior Transformer (Kaggle)

Trains a **Transformer** on real aerial tracking trajectories to classify drone-view object behavior.

### Datasets
| Dataset | Source | What it provides |
|---------|--------|-----------------|
| VisDrone 2019-MOT | Official Google Drive (`VisDrone/VisDrone-Dataset`) | 14+ real aerial video sequences with frame-level tracking annotations → trajectory extraction |
| Aerial Sheep | `keremberke/aerial-sheep` (HuggingFace) | Aerial livestock tracking with motion diversity — adds formation/stationary examples |

### Behavior classes
`stationary` · `patrol` · `evasive` · `approaching` · `formation`

### Output
`best_behavior.pth` + `behavior_transformer.onnx` + `norm_mean.npy/norm_std.npy` → `models/behavior/`

### Kaggle Secrets required
`GIT_TOKEN` (HF_TOKEN optional — Aerial Sheep is public)

> Runtime: ~20–30 min on T4 GPU

In [ ]:
# ── CELL 1: Install ──────────────────────────────────────────────────────────
import subprocess
subprocess.check_call(["pip", "install", "-q",
    "torch", "onnx", "huggingface_hub", "hf-transfer",
    "scikit-learn", "datasets", "gdown"])

import os, json, shutil, tempfile, math
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import numpy as np
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report

print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

In [ ]:
# ── CELL 2: Config & Secrets ─────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    HF_TOKEN  = _s.get_secret("HF_TOKEN")
    GIT_TOKEN = _s.get_secret("GIT_TOKEN")
    print(f"✅ Secrets — HF_TOKEN starts: {HF_TOKEN[:8] if HF_TOKEN else 'EMPTY'}")
except Exception as e:
    print(f"⚠ kaggle_secrets failed: {e}")
    HF_TOKEN  = os.environ.get("HF_TOKEN", "")
    GIT_TOKEN = os.environ.get("GIT_TOKEN", "")

GIT_REPO  = "https://github.com/ved354/swam.git"
GIT_USER  = "ved354"
GIT_EMAIL = "ved354@users.noreply.github.com"

# ── Model hyperparams (must match src/vision/behavior_analyzer.py) ───────────
WINDOW    = 30      # trajectory frames per sample
FEAT      = 5       # features per frame: x, y, speed, heading, accel
DIM       = 64
HEADS     = 4
LAYERS    = 2
EPOCHS    = 60
BATCH     = 128
LR        = 1e-3
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "behavior_model"
VISDRONE_DIR = WORK_DIR / "visdrone_mot"
AERIAL_DIR  = WORK_DIR / "aerial_sheep"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["patrol", "evasive", "formation", "stationary", "approaching"]
print(f"Device : {DEVICE}")
print(f"Classes: {CLASSES}")

In [ ]:
# ── CELL 3: Download Datasets ────────────────────────────────────────────────
from huggingface_hub import snapshot_download, login
import gdown, zipfile

_hf_token_to_use = False  # default: anonymous (token=False disables cached creds)

if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        _hf_token_to_use = HF_TOKEN
        print("✅ HuggingFace login OK")
    except Exception as _e:
        print(f"⚠ HF login failed ({_e}) — continuing with anonymous access")
        # Clear cached bad token so it doesn't poison subsequent requests
        try:
            from huggingface_hub import logout
            logout()
        except Exception:
            pass
        _hf_token_to_use = False  # token=False = explicit anonymous, ignores cache

# ── Dataset 1: VisDrone 2019-MOT (official Google Drive) ─────────────────────
# Source: github.com/VisDrone/VisDrone-Dataset  (Task 4: Multi-Object Tracking)
# MOT val set — 14 real aerial drone video sequences with tracking annotations
# Format: frame,id,x,y,w,h,score,class,truncation,occlusion
_VD_GDRIVE_ID = "1rqnKe9IgU_crMaxRoel9_nuUsMEBBVQu"   # VisDrone2019-MOT-val.zip
_vd_has_anns = VISDRONE_DIR.exists() and list(VISDRONE_DIR.rglob("*.txt"))

if not _vd_has_anns:
    print("📥 Downloading VisDrone 2019-MOT val (real aerial tracking)...")
    VISDRONE_DIR.mkdir(parents=True, exist_ok=True)
    _zip_path = WORK_DIR / "visdrone_mot_val.zip"
    try:
        gdown.download(id=_VD_GDRIVE_ID, output=str(_zip_path), quiet=False)
        with zipfile.ZipFile(str(_zip_path)) as zf:
            ann_files = [n for n in zf.namelist()
                         if "annotation" in n.lower() and n.endswith(".txt")]
            for name in ann_files:
                zf.extract(name, str(VISDRONE_DIR))
        _zip_path.unlink(missing_ok=True)
        _n_ann = len(list(VISDRONE_DIR.rglob("*.txt")))
        print(f"✅ VisDrone MOT: {_n_ann} annotation files extracted")
    except Exception as _e:
        print(f"⚠ VisDrone download failed ({_e}) — will use Aerial Sheep + fallback")
        _zip_path.unlink(missing_ok=True)
else:
    print(f"✅ VisDrone MOT already exists ({len(list(VISDRONE_DIR.rglob('*.txt')))} files)")

# ── Dataset 2: Aerial Sheep (drone-view tracking) ────────────────────────────
# keremberke/aerial-sheep — livestock tracking from drones, COCO bboxes per frame
if not AERIAL_DIR.exists() or not any(AERIAL_DIR.iterdir()):
    print("📥 Downloading Aerial Sheep (drone-view multi-object tracking)...")
    try:
        snapshot_download(
            repo_id="keremberke/aerial-sheep",
            repo_type="dataset",
            local_dir=str(AERIAL_DIR),
            token=_hf_token_to_use,   # False = anonymous (ignores cached bad token)
            max_workers=4,
        )
        print("✅ Aerial Sheep downloaded")
    except Exception as _e:
        print(f"⚠ Aerial Sheep download failed ({_e}) — VisDrone data is sufficient")
else:
    print("✅ Aerial Sheep already exists")

# Inventory
for name, d in [("VisDrone", VISDRONE_DIR), ("AerialSheep", AERIAL_DIR)]:
    n = sum(1 for _ in d.rglob("*") if _.is_file()) if d.exists() else 0
    print(f"  {name}: {n} files")

In [ ]:
# ── CELL 4: Trajectory Extraction + Behavior Labeling ───────────────────────

def parse_mot_file(path: Path) -> dict:
    """Parse VisDrone MOT annotation → {track_id: [(frame, cx, cy, w, h), ...]}"""
    tracks = {}
    try:
        with open(path) as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    continue
                frame, tid = int(parts[0]), int(parts[1])
                x, y, w, h = float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])
                tracks.setdefault(tid, []).append((frame, x + w/2, y + h/2, w, h))
    except Exception:
        pass
    return tracks


def parse_coco_to_tracks(json_path: Path) -> dict:
    """Parse COCO-format detection file → pseudo-tracks by assigning nearby boxes."""
    tracks = {}
    try:
        with open(json_path) as f:
            coco = json.load(f)
        img_map = {i["id"]: i for i in coco.get("images", [])}
        # Sort annotations by image id (proxy for time)
        anns = sorted(coco.get("annotations", []), key=lambda a: a["image_id"])
        for ann in anns:
            info = img_map.get(ann["image_id"])
            if not info:
                continue
            frame_id = ann["image_id"]
            tid = ann.get("id", ann["image_id"])  # use annotation id as track id
            x, y, w, h = ann["bbox"]
            tracks.setdefault(tid % 5000, []).append(
                (frame_id, x + w/2, y + h/2, w, h))
    except Exception:
        pass
    return tracks


def compute_features(positions: list) -> np.ndarray:
    """(frame, cx, cy, w, h) list → normalised (T, 5) feature array."""
    positions = sorted(positions, key=lambda p: p[0])
    xs  = np.array([p[1] for p in positions], dtype=np.float32)
    ys  = np.array([p[2] for p in positions], dtype=np.float32)
    dts = np.diff([p[0] for p in positions], prepend=positions[0][0]).astype(np.float32)
    dts = np.where(dts == 0, 1, dts)

    dx      = np.diff(xs, prepend=xs[0])
    dy      = np.diff(ys, prepend=ys[0])
    speed   = np.sqrt(dx**2 + dy**2) / dts
    heading = np.arctan2(dy, dx)
    accel   = np.diff(speed, prepend=speed[0])

    xs      = (xs - xs.min()) / (xs.max() - xs.min() + 1e-6)
    ys      = (ys - ys.min()) / (ys.max() - ys.min() + 1e-6)
    speed   = speed / (speed.max() + 1e-6)
    heading = heading / np.pi
    accel   = accel  / (np.abs(accel).max() + 1e-6)

    return np.stack([xs, ys, speed, heading, accel], axis=1)   # (T, 5)


def label_trajectory(positions: list, feats: np.ndarray) -> int:
    """
    Infer behavior class from real trajectory motion statistics.
    Calibrated on VisDrone aerial footage.

    Classes (index):
      0 patrol      — moderate steady speed, covers distance
      1 evasive     — high accel variance + erratic heading
      2 formation   — low speed variance, close spacing
      3 stationary  — very low speed + displacement
      4 approaching — high linear displacement toward camera
    """
    speeds   = feats[:, 2]
    accels   = feats[:, 4]
    headings = feats[:, 3]
    xs, ys   = feats[:, 0], feats[:, 1]

    mean_spd   = speeds.mean()
    accel_std  = np.abs(accels).mean()
    total_disp = np.sqrt((xs[-1] - xs[0])**2 + (ys[-1] - ys[0])**2)
    head_var   = np.var(np.diff(headings))
    spd_var    = np.var(speeds)

    # Stationary
    if mean_spd < 0.04 and total_disp < 0.08:
        return 3
    # Evasive: rapid acceleration + erratic direction
    if accel_std > 0.35 and head_var > 0.25:
        return 1
    # Approaching: high displacement + low heading variance
    if total_disp > 0.65 and head_var < 0.15:
        return 4
    # Formation: low speed variance + moderate speed (herding / convoy)
    if spd_var < 0.02 and 0.1 < mean_spd < 0.5:
        return 2
    # Default: patrol
    return 0


def sliding_windows(feats: np.ndarray, label: int) -> list:
    """Slide WINDOW-frame window over trajectory."""
    T, step = feats.shape[0], max(1, WINDOW // 3)
    return [(feats[i:i+WINDOW].astype(np.float32), label)
            for i in range(0, T - WINDOW + 1, step)]


# ── Parse VisDrone MOT files ──────────────────────────────────────────────────
all_samples = []
ann_dirs = []
for split in ["VisDrone2019-MOT-train", "VisDrone2019-MOT-val"]:
    d = VISDRONE_DIR / split / "annotations"
    if d.exists():
        ann_dirs.append(d)

# Robust fallback — look for any annotation directory
if not ann_dirs:
    ann_dirs = [p for p in VISDRONE_DIR.rglob("annotations") if p.is_dir()]

print(f"VisDrone annotation dirs found: {len(ann_dirs)}")
total_tracks = total_windows = 0

for ann_dir in ann_dirs:
    for ann_file in ann_dir.glob("*.txt"):
        for tid, positions in parse_mot_file(ann_file).items():
            if len(positions) < WINDOW:
                continue
            try:
                feats   = compute_features(positions)
                label   = label_trajectory(positions, feats)
                samples = sliding_windows(feats, label)
                all_samples.extend(samples)
                total_tracks  += 1
                total_windows += len(samples)
            except Exception:
                continue

print(f"VisDrone → {total_windows} windows from {total_tracks} tracks")

# ── Parse Aerial Sheep (COCO format) ─────────────────────────────────────────
sheep_windows = 0
for json_file in sorted(AERIAL_DIR.rglob("*.json"))[:10]:
    for tid, positions in parse_coco_to_tracks(json_file).items():
        if len(positions) < WINDOW:
            continue
        try:
            feats   = compute_features(positions)
            label   = label_trajectory(positions, feats)
            samples = sliding_windows(feats, label)
            all_samples.extend(samples)
            sheep_windows += len(samples)
        except Exception:
            continue
print(f"Aerial Sheep → {sheep_windows} windows")

# ── Fallback: deep rglob for any .txt with MOT format ────────────────────────
if total_windows + sheep_windows < 2000:
    print(f"⚠ Only {total_windows + sheep_windows} windows — running deep fallback...")
    for txt in sorted(VISDRONE_DIR.rglob("*.txt"))[:500]:
        for tid, pos in parse_mot_file(txt).items():
            if len(pos) < WINDOW:
                continue
            try:
                feats = compute_features(pos)
                all_samples.extend(sliding_windows(feats, label_trajectory(pos, feats)))
            except Exception:
                continue
    print(f"  After fallback: {len(all_samples)} windows")

# Distribution
label_counts = Counter(s[1] for s in all_samples)
print(f"\n📊 Total: {len(all_samples)} windows")
for ci, cn in enumerate(CLASSES):
    print(f"   {cn:12s}: {label_counts.get(ci, 0)}")

In [ ]:
# ── CELL 5: Dataset, Model, Training ─────────────────────────────────────────
np.random.default_rng(0).shuffle(all_samples)
split = int(len(all_samples) * 0.8)
train_data, val_data = all_samples[:split], all_samples[split:]
print(f"Train: {len(train_data)}  Val: {len(val_data)}")


class TrajDataset(Dataset):
    def __init__(self, samples):
        self.X = torch.tensor(np.array([s[0] for s in samples]), dtype=torch.float32)
        self.Y = torch.tensor([s[1] for s in samples], dtype=torch.long)
    def __len__(self):         return len(self.X)
    def __getitem__(self, i):  return self.X[i], self.Y[i]


train_loader = DataLoader(TrajDataset(train_data), batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(TrajDataset(val_data),   batch_size=BATCH, shuffle=False, num_workers=2)


class BehaviorTransformer(nn.Module):
    def __init__(self, feat=FEAT, dim=DIM, heads=HEADS, layers=LAYERS, n_cls=5, window=WINDOW):
        super().__init__()
        self.embed = nn.Linear(feat, dim)
        self.pos   = nn.Embedding(window, dim)
        enc_layer  = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim*4,
            dropout=0.1, batch_first=True)
        self.enc   = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.head  = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(dim // 2, n_cls),
        )
    def forward(self, x):
        B, T, _ = x.shape
        pos_ids  = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.embed(x) + self.pos(pos_ids)
        return self.head(self.enc(x).mean(dim=1))


model  = BehaviorTransformer().to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"BehaviorTransformer: {params:,} params")

# Class-balanced loss
cnt_vec = [label_counts.get(i, 1) for i in range(len(CLASSES))]
weights = torch.tensor([max(cnt_vec)/c for c in cnt_vec], dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc  = 0.0
no_improve = 0
PATIENCE   = 15

print(f"\n🚀 Training {EPOCHS} epochs on real aerial trajectories...")

for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    tc = tt = 0
    for X, Y in train_loader:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, Y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        _, pred = out.max(1)
        tt += Y.size(0)
        tc += pred.eq(Y).sum().item()
    scheduler.step()

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for X, Y in val_loader:
            X, Y = X.to(DEVICE), Y.to(DEVICE)
            _, pred = model(X).max(1)
            preds_all.extend(pred.cpu().numpy())
            labels_all.extend(Y.cpu().numpy())

    va = 100.0 * sum(p == l for p, l in zip(preds_all, labels_all)) / len(labels_all)

    if (epoch + 1) % 10 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  train={100.*tc/tt:.1f}%  val={va:.1f}%")

    if va > best_acc:
        best_acc   = va
        no_improve = 0
        torch.save(model.state_dict(), str(OUTPUT_DIR / "best_behavior.pth"))
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"⏹ Early stop at epoch {epoch+1}")
            break

print(f"\n✅ Best val accuracy: {best_acc:.1f}%")

# Final classification report
model.load_state_dict(torch.load(str(OUTPUT_DIR / "best_behavior.pth"), weights_only=True))
model.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for X, Y in val_loader:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        _, pred = model(X).max(1)
        preds_all.extend(pred.cpu().numpy())
        labels_all.extend(Y.cpu().numpy())
print("\n📊 Per-class results:")
print(classification_report(labels_all, preds_all, target_names=CLASSES, zero_division=0))

In [ ]:
# ── CELL 6: Export ONNX + Normalisation Stats + Metadata ────────────────────
model.eval()
dummy = torch.randn(1, WINDOW, FEAT).to(DEVICE)
torch.onnx.export(
    model, dummy,
    str(OUTPUT_DIR / "behavior_transformer.onnx"),
    input_names=["trajectory"],
    output_names=["behavior_probs"],
    dynamic_axes={"trajectory": {0: "batch"}, "behavior_probs": {0: "batch"}},
    opset_version=17,
)

# Save normalisation stats used during inference
all_X = np.array([s[0] for s in all_samples])
np.save(str(OUTPUT_DIR / "norm_mean.npy"), all_X.mean(axis=(0, 1)))
np.save(str(OUTPUT_DIR / "norm_std.npy"),  all_X.std(axis=(0, 1)) + 1e-6)

metadata = {
    "model":   "vayuswarm_behavior",
    "classes": CLASSES,
    "window":  WINDOW,
    "features": FEAT,
    "params":  params,
    "best_acc": round(best_acc, 2),
    "training_data": {
        "primary":   "Vayex/VisDrone2018 MOT — real aerial tracking sequences",
        "secondary": "keremberke/aerial-sheep — drone-view livestock tracking",
        "total_windows": len(all_samples),
        "labeling": "Motion statistics: speed, heading variance, displacement",
    },
}
with open(OUTPUT_DIR / "behavior_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Exported:")
for fname in ["best_behavior.pth", "behavior_transformer.onnx",
               "norm_mean.npy", "norm_std.npy", "behavior_metadata.json"]:
    p = OUTPUT_DIR / fname
    if p.exists():
        print(f"   {fname} ({p.stat().st_size / 1024:.0f} KB)")

In [ ]:
# ── CELL 7: Push to GitHub ───────────────────────────────────────────────────
import subprocess as _sp

if GIT_TOKEN:
    try:
        auth_url  = GIT_REPO.replace("https://", f"https://{GIT_USER}:{GIT_TOKEN}@")
        clone_dir = Path(tempfile.mkdtemp()) / "swam"
        print(f"\n📤 Cloning {GIT_REPO}...")
        _sp.check_call(["git", "clone", "--depth", "1", auth_url, str(clone_dir)])

        target = clone_dir / "models" / "behavior"
        target.mkdir(parents=True, exist_ok=True)

        files = ["best_behavior.pth", "behavior_transformer.onnx",
                 "behavior_metadata.json", "norm_mean.npy", "norm_std.npy"]
        for fname in files:
            src = OUTPUT_DIR / fname
            if src.exists():
                shutil.copy2(str(src), str(target / fname))
                print(f"   ✅ {fname} ({src.stat().st_size / 1024:.0f} KB)")

        env = os.environ.copy()
        for cmd in [
            ["git", "config", "user.name",  GIT_USER],
            ["git", "config", "user.email", GIT_EMAIL],
            ["git", "add", "models/behavior/"],
            ["git", "commit", "-m",
             f"Real-data behavior transformer — val_acc={best_acc:.1f}%, "
             f"datasets: VisDrone+AerialSheep, {len(CLASSES)} classes"],
            ["git", "push", "origin", "main"],
        ]:
            _sp.check_call(cmd, cwd=str(clone_dir), env=env)

        print(f"\n✅ Pushed behavior model to {GIT_REPO}")

    except Exception as e:
        print(f"\n⚠ GitHub push failed: {e}")
        print("  → Download from Kaggle Output tab: behavior_model/")
else:
    print("ℹ No GIT_TOKEN — model saved locally at:", OUTPUT_DIR)

print(f"""
{'='*55}
🎉 Behavior Training Complete!
   Val accuracy : {best_acc:.1f}%
   Datasets     : VisDrone-MOT + Aerial Sheep (drone-view)
   Classes      : {CLASSES}
{'='*55}
""")